# Anatomía de una Falla · Parte II — 03 · Líneas 2 y 3 (experimentos exploratorios)

- **Línea 2 · Generalizar entre tipos de vuelo:** ¿un modelo entrenado sólo con vuelos cortos estima bien la degradación
  en vuelos medianos y largos? ¿Ayuda descontar el efecto operativo (residuos) frente a usar sensores crudos?
- **Línea 3 · Detección temprana:** ¿se puede detectar el inicio de la degradación (cambio de régimen) antes de que la
  vida útil caiga, sólo con los sensores? Se compara contra el inicio real (`hs`).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from src.ncmapss import nb
MODE = "full"          # "sample" si corriste el pipeline con --sample

In [2]:
L2 = nb.load('line2', MODE)
L3 = nb.load('line3', MODE)
import pandas as pd
pd.set_option('display.max_colwidth', None)

## Línea 2 · De vuelos cortos a vuelos largos

In [3]:
pd.DataFrame(L2['line2_table'])

,variante,motores de entrenamiento,RMSE vuelos cortos,RMSE vuelos medianos,RMSE vuelos largos,NASA vuelos cortos,NASA vuelos medianos,NASA vuelos largos
0,A · Todas las clases · residuos,60,5.45,6.60,7.61,0.588,0.713,0.866
1,"E · Control: igual cantidad de motores, todas las clases",14,7.91,9.28,9.25,1.106,1.411,1.477
2,B · Sólo cortos · sensores crudos,14,7.86,17.10,18.24,1.122,5.450,6.058
3,C · Sólo cortos · residuos (modelo sano de cortos),14,8.33,15.46,14.70,2.138,5.827,5.551
4,D · Sólo cortos · residuos (modelo sano general),14,8.21,17.09,16.97,1.960,6.997,6.955


In [4]:
nb.show('line2', L2)

> **Lectura.** En vuelos largos, el modelo con todos los motores de desarrollo tiene RMSE 7,61. Con sólo 14 motores sorteados entre todas las clases (control E) sube a 9,25: **1,6 vuelos** se explican por tener menos datos. Con 14 motores de vuelos _cortos_ (C) llega a 14,70: **5,4 vuelos más** son el costo de no haber visto ese tipo de vuelo. Usar el modelo sano general en lugar del de cortos (D vs. C) no mejora el resultado en vuelos largos (16,97 contra 14,70), y descontar el efecto operativo ayuda frente a usar sensores crudos (B: 18,24). El modelo sano de cortos explica el 99,998 % de un sensor en vuelos cortos y el 99,13 % en vuelos largos.

## Línea 3 · Detección temprana con change points

In [5]:
pd.DataFrame(L3['line3_table'])

,método,motores,detectados [%],alarma antes del inicio real [%],demora mediana [vuelos],anticipación mediana a la falla [vuelos],desgaste ocurrido al alarmar [%]
0,CUSUM online,39,100.0,10.3,6.0,43.0,3.8
1,PELT offline,39,100.0,25.6,4.0,46.0,2.3
2,Umbral sobre T48,39,94.9,15.4,7.0,38.0,4.5


In [6]:
nb.show('line3_delay', L3)

> **Lectura.** Con cada umbral calibrado en motores de desarrollo para no superar 10 % de falsas alarmas, en los 39 motores de prueba el CUSUM online da la alarma con una demora mediana de **6 vuelos** respecto del inicio real, cuando falta una mediana de 43 vuelos para la falla y sólo ocurrió el **4 %** del desgaste total. El umbral sobre T48 sola tarda 7 vuelos (4 % del desgaste). PELT, que mira la serie completa, marca 4. Alarmas antes del inicio real en prueba: CUSUM 10 %, T48 15 %, PELT 26 %.

In [7]:
nb.show('line3_example', L3)

> **Lectura.** El índice de salud resume los 14 residuos en un número: plano mientras el motor está sano y creciente cuando empieza la degradación. El CUSUM acumula los excesos sobre lo normal y da la alarma (línea de color) cuando la acumulación supera el umbral.

## Lectura y trabajo futuro

In [8]:
for t in L2['future_discussion_l2'] + L3['future_discussion_l3']: nb.html('<p>'+t+'</p>')